In [ ]:
# Import required packages
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

In [ ]:
# Directories
CODE_DIR = Path(r"D:\StockTwits\Code")
DATA_DIR = Path(r"D:\StockTwits\Data\v1\data\csv")
FIGURES_DIR = Path(r"D:\StockTwits\Figures")
MODEL_DATA_DIR = Path(r"D:\StockTwits\Data")

# File names
INPUT_DATA = MODEL_DATA_DIR / "merged_master.pkl"

# Sample period
SAMPLE_START = '2012-01-01'
SAMPLE_END = '2022-12-31'

# Legacy text-variant switch (the add_text_features builder it referred to was removed on 2026-09-11):
#   None -> the plain all-features predictions; "raw" | "pca" | "supervised" -> the matching
#   predictions_*_input=N_text=<variant>.pkl files written by a model notebook run with that TEXT_VARIANT.
TEXT_VARIANT = None

def find_all_features_file(model_type, text_variant=None):
    """Resolve the 'all features' prediction filename for a model type: the largest
    input-count file on disk (excluding the 2-feature baseline) whose _text=<variant> tag
    matches `text_variant` (files without a tag when text_variant is None), so this doesn't
    need updating whenever the feature set changes."""
    import re
    pattern = re.compile(r"input=(\d+)(?:_text=([A-Za-z]+))?")
    candidates = []
    for p in MODEL_DATA_DIR.glob(f"predictions_{model_type}_input=*.pkl"):
        m = pattern.search(p.stem)
        if m is None or int(m.group(1)) == 2 or m.group(2) != text_variant:
            continue
        candidates.append((int(m.group(1)), p.name))
    if not candidates:
        tag = "" if text_variant is None else f"_text={text_variant}"
        return f"predictions_{model_type}_input=NOT_FOUND{tag}.pkl"
    return max(candidates, key=lambda t: t[0])[1]

# Model registry: add new models here
# Key = column name in the dataframe, Value = prediction filename
# Evaluate every registered model on the same stock-days: rows where any model has no
# prediction are dropped. The text-only model predicts only stock-days with messages, so
# with it registered the common sample is (essentially) the tweeted stock-days.
COMMON_SAMPLE = True

MODELS = {
    'lr_2': 'predictions_linear_regression_input=2.pkl',
    'lr_all': find_all_features_file('linear_regression', TEXT_VARIANT),
    # Text-only OLS on the 384 embedding dimensions (03a/prediction_linear_regression_text_only.ipynb).
    # Distinct model name on purpose: the resolver above never picks it up, so it is registered explicitly.
    'lr_text': 'predictions_linear_regression_textonly_input=384.pkl',
}

In [ ]:
# Load base data (actual returns + raw_volume to identify no-tweet observations)
TARGET = 'f_cumret1'
df = pd.read_pickle(INPUT_DATA)[['date', 'permno', 'ticker', TARGET, 'log_volume']].copy()

# Merge predictions from each model
for model_name, pred_file in MODELS.items():
    preds = pd.read_pickle(MODEL_DATA_DIR / pred_file)
    df[model_name] = preds.set_index('index')['prediction']

# Keep only rows with at least one prediction (OOS period)
model_cols = list(MODELS.keys())
df = df.dropna(subset=model_cols, how='all').reset_index(drop=True)

# Add time period columns and restrict to sample period
df['date'] = pd.to_datetime(df['date'])
df = df[(df['date'] >= SAMPLE_START) & (df['date'] <= SAMPLE_END)].reset_index(drop=True)
df['year_month'] = df['date'].dt.to_period('M')
df['year'] = df['date'].dt.year

print(f"OOS sample: {len(df):,} rows")
print(f"Period: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Models loaded: {model_cols}")
for col in model_cols:
    print(f"  {col}: {df[col].notna().sum():,} predictions")

In [ ]:
# Drop stock-days with no tweets (log_volume == 0 means no StockTwits messages)
n_before = len(df)
df = df[df['log_volume'] > 0].reset_index(drop=True)
print(f"Dropped {n_before - len(df):,} no-tweet observations ({(n_before - len(df)) / n_before:.1%})")
print(f"Remaining: {len(df):,} rows")

if COMMON_SAMPLE:
    n_before = len(df)
    df = df.dropna(subset=model_cols).reset_index(drop=True)
    print(f"COMMON_SAMPLE: dropped {n_before - len(df):,} rows lacking a prediction from some model; remaining {len(df):,}")

In [ ]:
# Cross-sectionally de-mean the target and predictions each day
daily_means = df.groupby('date')[[TARGET] + model_cols].transform('mean')
df[TARGET] = df[TARGET] - daily_means[TARGET]
for col in model_cols:
    df[col] = df[col] - daily_means[col]

print("De-meaned target and predictions by date")
print(f"  Mean of {TARGET} after de-meaning: {df[TARGET].mean():.2e}")
for col in model_cols:
    print(f"  Mean of {col} after de-meaning: {df[col].mean():.2e}")

# OOS MSE and R-squared

In [ ]:
def compute_metrics(df, group_col, model_cols, target='f_cumret1'):
    """Compute MSE and OOS R-squared for each model, grouped by period."""
    results = []
    for name, g in df.groupby(group_col):
        row = {'period': name, 'N': len(g)}
        for col in model_cols:
            valid = g[[target, col]].dropna()
            if valid.empty:
                # f_cumret1 is a nullable Float64 column: an empty slice's .mean()/.sum() returns
                # pd.NA rather than np.nan, and one pd.NA turns the whole result column into
                # object dtype (which later breaks the matplotlib float conversion).
                row[f'MSE_{col}'] = np.nan
                row[f'R2_{col}'] = np.nan
                continue
            se = (valid[target] - valid[col]) ** 2
            ss_tot = ((valid[target] - valid[target].mean()) ** 2).sum()
            row[f'MSE_{col}'] = float(se.mean())
            row[f'R2_{col}'] = float(1 - se.sum() / ss_tot) if ss_tot > 0 else np.nan
        results.append(row)
    return pd.DataFrame(results).set_index('period')

In [ ]:
# Monthly MSE and R-squared
monthly = compute_metrics(df, 'year_month', model_cols)

print("Monthly OOS MSE and R-squared")
print("=" * 80)
print(monthly.to_string())

In [ ]:
# Yearly MSE and R-squared
yearly = compute_metrics(df, 'year', model_cols)

print("Yearly OOS MSE and R-squared")
print("=" * 80)
print(yearly.to_string())

In [ ]:
# Full sample MSE and R-squared
full = {}
for col in model_cols:
    valid = df[[TARGET, col]].dropna()
    se = (valid[TARGET] - valid[col]) ** 2
    ss_tot = ((valid[TARGET] - valid[TARGET].mean()) ** 2).sum()
    full[col] = {'MSE': se.mean(), 'R2_OOS': 1 - se.sum() / ss_tot, 'N': len(valid)}

full_sample = pd.DataFrame(full).T
full_sample.index.name = 'model'

print("Full Sample OOS Results")
print("=" * 50)
print(full_sample.to_string())

# OOS MSE and R-squared by prediction decile

In [ ]:
# Assign daily cross-sectional prediction deciles for each model.
# A date with fewer than 10 valid predictions for a model (e.g. a coverage gap) cannot be
# split into deciles -- leave it NaN rather than crash on "Bin edges must be unique" or
# silently form fewer than 10 bins.
def daily_deciles(x, q=10):
    if x.notna().sum() < q:
        return pd.Series(np.nan, index=x.index)
    return pd.qcut(x.rank(method='first'), q, labels=False) + 1

for col in model_cols:
    df[f'decile_{col}'] = df.groupby('date')[col].transform(daily_deciles)

In [ ]:
# Full sample MSE and R-squared by prediction decile
for col in model_cols:
    decile_metrics = compute_metrics(df, f'decile_{col}', [col])
    decile_metrics.index.name = 'decile'
    decile_metrics.columns = ['N', 'MSE', 'R2_OOS']
    print(f"Full sample OOS metrics by prediction decile: {col}")
    print("=" * 50)
    print(decile_metrics.to_string())
    print()

In [ ]:
# Yearly MSE and R-squared by prediction decile
for col in model_cols:
    decile_col = f'decile_{col}'
    results = []
    for (decile, year), g in df.groupby([decile_col, 'year']):
        valid = g[[TARGET, col]].dropna()
        if valid.empty:
            continue
        se = (valid[TARGET] - valid[col]) ** 2
        ss_tot = ((valid[TARGET] - valid[TARGET].mean()) ** 2).sum()
        results.append({
            'decile': int(decile), 'year': year,
            'MSE': se.mean(),
            'R2_OOS': 1 - se.sum() / ss_tot if ss_tot > 0 else np.nan,
        })
    res = pd.DataFrame(results)
    print(f"Yearly MSE by prediction decile: {col}")
    print("=" * 100)
    print(res.pivot(index='year', columns='decile', values='MSE').to_string())
    print(f"\nYearly R-squared by prediction decile: {col}")
    print("=" * 100)
    print(res.pivot(index='year', columns='decile', values='R2_OOS').to_string())
    print()

In [ ]:
# Monthly MSE and R-squared by prediction decile
for col in model_cols:
    decile_col = f'decile_{col}'
    results = []
    for (decile, period), g in df.groupby([decile_col, 'year_month']):
        valid = g[[TARGET, col]].dropna()
        if valid.empty:
            continue
        se = (valid[TARGET] - valid[col]) ** 2
        ss_tot = ((valid[TARGET] - valid[TARGET].mean()) ** 2).sum()
        results.append({
            'decile': int(decile), 'period': period,
            'MSE': se.mean(),
            'R2_OOS': 1 - se.sum() / ss_tot if ss_tot > 0 else np.nan,
        })
    res = pd.DataFrame(results)
    print(f"Monthly MSE by prediction decile: {col}")
    print("=" * 100)
    print(res.pivot(index='period', columns='decile', values='MSE').to_string())
    print(f"\nMonthly R-squared by prediction decile: {col}")
    print("=" * 100)
    print(res.pivot(index='period', columns='decile', values='R2_OOS').to_string())
    print()

In [ ]:
# Yearly MSE by prediction decile (5x2 subplots, one per decile)
fig, axes = plt.subplots(5, 2, figsize=(16, 20), sharex=True, sharey=True)
fig.suptitle('Yearly MSE by Prediction Decile', fontsize=14, y=1.0)
for d in range(1, 11):
    ax = axes[(d - 1) // 2, (d - 1) % 2]
    for col in model_cols:
        subset = df[df[f'decile_{col}'] == d]
        yearly_mse = subset.groupby('year').apply(
            lambda g: ((g[TARGET] - g[col]) ** 2).mean()
        )
        ax.plot(yearly_mse.index, yearly_mse.values, marker='o', markersize=4, label=col)
    ax.set_title(f'Decile {d}', fontsize=10)
    ax.grid(True, alpha=0.3)
    if d == 1:
        ax.legend(fontsize=8)
    if (d - 1) % 2 == 0:
        ax.set_ylabel('MSE')
    if d >= 9:
        ax.set_xlabel('Year')
plt.tight_layout()
plt.show()

In [ ]:
# Yearly R-squared by prediction decile (5x2 subplots, one per decile)
fig, axes = plt.subplots(5, 2, figsize=(16, 20), sharex=True, sharey=True)
fig.suptitle('Yearly R-squared by Prediction Decile', fontsize=14, y=1.0)
for d in range(1, 11):
    ax = axes[(d - 1) // 2, (d - 1) % 2]
    for col in model_cols:
        subset = df[df[f'decile_{col}'] == d]
        yearly_r2 = subset.groupby('year').apply(
            lambda g: 1 - ((g[TARGET] - g[col]) ** 2).sum() / ((g[TARGET] - g[TARGET].mean()) ** 2).sum()
        )
        ax.plot(yearly_r2.index, yearly_r2.values, marker='o', markersize=4, label=col)
    ax.axhline(y=0, color='black', linestyle='--', linewidth=0.8)
    ax.set_title(f'Decile {d}', fontsize=10)
    ax.grid(True, alpha=0.3)
    if d == 1:
        ax.legend(fontsize=8)
    if (d - 1) % 2 == 0:
        ax.set_ylabel('R-squared')
    if d >= 9:
        ax.set_xlabel('Year')
plt.tight_layout()
plt.show()

In [ ]:
# Monthly MSE by prediction decile (5x2 subplots, one per decile)
fig, axes = plt.subplots(5, 2, figsize=(16, 20), sharex=True, sharey=True)
fig.suptitle('Monthly MSE by Prediction Decile', fontsize=14, y=1.0)
for d in range(1, 11):
    ax = axes[(d - 1) // 2, (d - 1) % 2]
    for col in model_cols:
        subset = df[df[f'decile_{col}'] == d]
        monthly_mse = subset.groupby('year_month').apply(
            lambda g: ((g[TARGET] - g[col]) ** 2).mean()
        )
        ax.plot(monthly_mse.index.to_timestamp(), monthly_mse.values, label=col, linewidth=0.8)
    ax.set_title(f'Decile {d}', fontsize=10)
    ax.grid(True, alpha=0.3)
    if d == 1:
        ax.legend(fontsize=8)
    if (d - 1) % 2 == 0:
        ax.set_ylabel('MSE')
    if d >= 9:
        ax.set_xlabel('Month')
plt.tight_layout()
plt.show()

In [ ]:
# Monthly R-squared by prediction decile (5x2 subplots, one per decile)
fig, axes = plt.subplots(5, 2, figsize=(16, 20), sharex=True, sharey=True)
fig.suptitle('Monthly R-squared by Prediction Decile', fontsize=14, y=1.0)
for d in range(1, 11):
    ax = axes[(d - 1) // 2, (d - 1) % 2]
    for col in model_cols:
        subset = df[df[f'decile_{col}'] == d]
        monthly_r2 = subset.groupby('year_month').apply(
            lambda g: 1 - ((g[TARGET] - g[col]) ** 2).sum() / ((g[TARGET] - g[TARGET].mean()) ** 2).sum()
        )
        ax.plot(monthly_r2.index.to_timestamp(), monthly_r2.values, label=col, linewidth=0.8)
    ax.axhline(y=0, color='black', linestyle='--', linewidth=0.8)
    ax.set_title(f'Decile {d}', fontsize=10)
    ax.grid(True, alpha=0.3)
    if d == 1:
        ax.legend(fontsize=8)
    if (d - 1) % 2 == 0:
        ax.set_ylabel('R-squared')
    if d >= 9:
        ax.set_xlabel('Month')
plt.tight_layout()
plt.show()

# Plots

In [ ]:
# Monthly MSE
fig, ax = plt.subplots(figsize=(14, 5))
x = monthly.index.to_timestamp()
for col in model_cols:
    ax.plot(x, monthly[f'MSE_{col}'], label=col)
ax.set_xlabel('Month')
ax.set_ylabel('MSE')
ax.set_title('Monthly OOS MSE')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Monthly R-squared
fig, ax = plt.subplots(figsize=(14, 5))
x = monthly.index.to_timestamp()
for col in model_cols:
    ax.plot(x, monthly[f'R2_{col}'], label=col)
ax.axhline(y=0, color='black', linestyle='--', linewidth=0.8)
ax.set_xlabel('Month')
ax.set_ylabel('R-squared')
ax.set_title('Monthly OOS R-squared')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Yearly MSE
fig, ax = plt.subplots(figsize=(10, 5))
x = yearly.index.astype(int)
for col in model_cols:
    ax.plot(x, yearly[f'MSE_{col}'], marker='o', label=col)
ax.set_xlabel('Year')
ax.set_ylabel('MSE')
ax.set_title('Yearly OOS MSE')
ax.set_xticks(x)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Yearly R-squared
fig, ax = plt.subplots(figsize=(10, 5))
x = yearly.index.astype(int)
for col in model_cols:
    ax.plot(x, yearly[f'R2_{col}'], marker='o', label=col)
ax.axhline(y=0, color='black', linestyle='--', linewidth=0.8)
ax.set_xlabel('Year')
ax.set_ylabel('R-squared')
ax.set_title('Yearly OOS R-squared')
ax.set_xticks(x)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()